# Tahim Bhuiya

# Logistic Regression & Regularization

In [1]:
import pandas as pd
df = pd.read_csv('Airline_Delay_Cause.csv')
df

,year,month,carrier,carrier_name,airport,airport_name,arr_flights,arr_del15,carrier_ct,weather_ct,...,security_ct,late_aircraft_ct,arr_cancelled,arr_diverted,arr_delay,carrier_delay,weather_delay,nas_delay,security_delay,late_aircraft_delay
0,2022,5,9E,Endeavor Air Inc.,ABE,"Allentown/Bethlehem/Easton, PA: Lehigh Valley ...",136.0,7.0,5.95,0.00,...,0.0,1.00,0.0,0.0,255.0,222.0,0.0,4.0,0.0,29.0
1,2022,5,9E,Endeavor Air Inc.,ABY,"Albany, GA: Southwest Georgia Regional",91.0,16.0,7.38,0.00,...,0.0,6.09,0.0,0.0,884.0,351.0,0.0,81.0,0.0,452.0
2,2022,5,9E,Endeavor Air Inc.,ACK,"Nantucket, MA: Nantucket Memorial",19.0,2.0,0.13,0.00,...,0.0,0.88,1.0,0.0,138.0,4.0,0.0,106.0,0.0,28.0
3,2022,5,9E,Endeavor Air Inc.,AEX,"Alexandria, LA: Alexandria International",88.0,14.0,7.26,0.76,...,0.0,1.64,0.0,0.0,947.0,585.0,35.0,125.0,0.0,202.0
4,2022,5,9E,Endeavor Air Inc.,AGS,"Augusta, GA: Augusta Regional at Bush Field",181.0,19.0,13.84,0.00,...,0.0,2.09,0.0,0.0,808.0,662.0,0.0,87.0,0.0,59.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
318012,2003,6,DL,Delta Air Lines Inc.,SEA,"Seattle, WA: Seattle/Tacoma International",480.0,84.0,25.69,3.09,...,0.0,21.26,0.0,0.0,3744.0,978.0,164.0,1023.0,0.0,1579.0
318013,2003,6,DL,Delta Air Lines Inc.,SFO,"San Francisco, CA: San Francisco International",505.0,111.0,21.78,2.24,...,0.0,13.17,3.0,0.0,4284.0,1376.0,138.0,2132.0,0.0,638.0
318014,2003,6,DL,Delta Air Lines Inc.,SJC,"San Jose, CA: Norman Y. Mineta San Jose Intern...",146.0,36.0,6.99,0.00,...,0.0,2.15,0.0,0.0,896.0,205.0,0.0,607.0,0.0,84.0
318015,2003,6,DL,Delta Air Lines Inc.,SJU,"San Juan, PR: Luis Munoz Marin International",95.0,13.0,3.66,0.00,...,0.0,1.42,0.0,0.0,367.0,120.0,0.0,210.0,0.0,37.0


In [2]:
df.columns # print the columns

Index(['year', 'month', 'carrier', 'carrier_name', 'airport', 'airport_name',
       'arr_flights', 'arr_del15', 'carrier_ct', 'weather_ct', 'nas_ct',
       'security_ct', 'late_aircraft_ct', 'arr_cancelled', 'arr_diverted',
       'arr_delay', 'carrier_delay', 'weather_delay', 'nas_delay',
       'security_delay', 'late_aircraft_delay'],
      dtype='object')

In [3]:
df.shape # print the number of rows and columns

(318017, 21)

In [4]:
df = df.dropna() # remove rows containing nan values

In [5]:
df = df.drop_duplicates(keep='first') # drop duplicated rows, keep the first occurrence

In [6]:
df.shape

(317261, 21)

# Defining the Prediction Task

## A. Categorical outcome variable 

Choosing a categorical outcome variable and in a markdown cell state:

• What the variable is

• What “1” means

• What “0” means





**Outcome:** delay_category – whether a flight has a high arrival delay  
- **1 = High delay** (flight’s arrival delay is above the median, arr_del15 > median)  
- **0 = Low delay** (flight’s arrival delay is at or below the median, arr_del15 <= median)  

This defines a binary classification problem: predicting whether a flight is likely to experience a high delay.

## B. Predictor variables 

List at least predictor variables and explaining why each one
may help predict the outcome.

1. **month** – Problems such as seasonal conditions can increase the likelihood of delays.  
2. **arr_flights** – Total flights at the airport may indicate congestion which can lead to delays.  
3. **carrier_ct** – Number of delays caused by the carrier. Higher counts can indicate issues.  
4. **weather_category** – Newly created high/low weather-delay category. This indicates high weather delays which can help in predicting. 
5. **nas_delay** – Delays due to the National Airspace System. This can affect arrival times.  
6. **late_aircraft_ct** – Delays caused by the aircraft arriving late from previous flights. This may be indicative of more delays.
7. **arr_delay** – This is the total delay time and can serve as strong indicator of delay severity.

In [7]:
import pandas as pd

# Create high/low delay category based on median of arr_del15
delay_threshold = df['arr_del15'].median()
df['delay_category'] = df['arr_del15'] > delay_threshold  # True = High delay, False = Low delay

# Crosstab of carrier_name vs delay_category
observations = pd.crosstab(index=df["carrier_name"], columns=df["delay_category"], margins=True)
observations

delay_category,False,True,All
carrier_name,,,
ATA Airlines d/b/a ATA,478,440,918
AirTran Airways Corporation,2630,4325,6955
Alaska Airlines Inc.,8614,4319,12933
Allegiant Air,5484,885,6369
Aloha Airlines Inc.,159,94,253
America West Airlines Inc.,668,911,1579
American Airlines Inc.,6357,13413,19770
American Eagle Airlines Inc.,5644,9755,15399
Atlantic Coast Airlines,512,741,1253


In [8]:
# Create high/low weather-delay category based on median of weather_ct that will be used as a feature
weather_threshold = df['weather_ct'].median()
df['weather_category'] = df['weather_ct'] > weather_threshold  # True = High weather delay, False = Low

# Data Preparation

Preparing the dataset for modeling:

• Converting categorical variables into numeric form 

• Handling missing values 

• Separating the data into X (features) and y (outcome) 

• Splitting the data into training and test sets 

• Balancing the training data 

In [9]:
from sklearn.model_selection import train_test_split, KFold, cross_val_score
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import mean_squared_error
from sklearn import preprocessing

## Converting categorical variables into numeric form

In [10]:
# Convert weather_category (True/False) → (1/0)
df['weather_category'] = df['weather_category'].astype(int)

## Handle missing values

Missing values (NaNs) have already been removed at beginning of notebook

Was done with:

df = df.dropna() # remove rows containing nan values

df = df.drop_duplicates(keep='first') # drop duplicated rows, keep the first occurrence

## Separate the data into X (features) and y (outcome)

In [11]:
# Select X and y
X = df[['month', 'arr_flights', 'carrier_ct', 'nas_delay', 
            'late_aircraft_ct', 'security_delay', 'weather_category']]
y = df['delay_category'].astype(int)

## Split the data into training and test sets

In [12]:
# Split data into training and testing sets (80% train, 20% test)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2)

## Balance the training data

In [13]:
train_df = pd.concat([X_train.reset_index(drop=True), y_train.reset_index(drop=True)], axis=1) # Combining to balance together

In [14]:
train_df['delay_category'].value_counts()

delay_category
0    130037
1    123771
Name: count, dtype: int64

In [ ]:
# 0 is majority, 1 is minority
majority_class = train_df[train_df['delay_category'] == 0]
minority_class = train_df[train_df['delay_category'] == 1]

# Undersample majority to match minority size
majority_class_undersampled = majority_class.sample(n=len(minority_class), random_state=42)

# Combine undersampled majority with full minority
train_balanced = pd.concat([majority_class_undersampled, minority_class])

 
